In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

# 1. Load the perfectly mixed synthetic dataset
df = pd.read_csv('../data/rw_classification_data.csv')

# 2. Safety Check: Drop duplicate math columns if they accidentally carried over
cols_to_drop = ['RW Calcium.1', 'RW Magnesium.1']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

# 3. Separate Features (Inputs) and Target (Output)
X = df_clean.drop(columns=['Water_Status'])
y = df_clean['Water_Status']

# 4. Train/Test Split (80% Train, 20% Test)
# stratify=y is CRITICAL here. It ensures the 58/42 ratio of Safe/Toxic is perfectly maintained in the test set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. SCALE THE DATA (Using RobustScaler to handle extreme synthetic spikes)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Classification Data Prepped and Scaled!")
print("-" * 50)
print(f"Tracking {X.shape[1]} Raw Water Metrics.")
print(f"Training Rows: {X_train_scaled.shape[0]} | Testing Rows: {X_test_scaled.shape[0]}")
print(f"Test Set Breakdown -> Safe: {sum(y_test==1)}, Toxic: {sum(y_test==0)}")

Classification Data Prepped and Scaled!
--------------------------------------------------
Tracking 15 Raw Water Metrics.
Training Rows: 577 | Testing Rows: 145
Test Set Breakdown -> Safe: 85, Toxic: 60


In [2]:
import numpy as np
import warnings
from sklearn.exceptions import ConvergenceWarning

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 1. Import the Classification Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier, 
    HistGradientBoostingClassifier
)
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# 2. Initialize the 8 Competitors
classifiers = {
    "Linear (Logistic)": LogisticRegression(max_iter=1000, random_state=42),
    "Distance (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Distance (SVM)": SVC(kernel='rbf', random_state=42),
    "Tree (Decision Tree)": DecisionTreeClassifier(random_state=42),
    "Tree (Random Forest)": RandomForestClassifier(n_estimators=100, random_state=42),
    "Boosting (Gradient)": GradientBoostingClassifier(random_state=42),
    "Boosting (Hist Gradient)": HistGradientBoostingClassifier(random_state=42),
    "Neural Net (MLP)": MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=2000, random_state=42)
}

print("🥊 THE CLASSIFICATION BAKE-OFF (CATCHING TOXIC WATER) 🥊\n" + "="*95)
print(f"{'Algorithm Name':<25} | {'Accuracy':<10} | {'Recall (Caught Toxins)':<25} | {'Precision (No False Alarms)'}")
print("-" * 95)

classification_results = {}

# 3. Train and Evaluate Loop
for name, clf in classifiers.items():
    try:
        # Train on SCALED inputs
        clf.fit(X_train_scaled, y_train)
        
        # Predict on unseen SCALED test data
        predictions = clf.predict(X_test_scaled)
        
        # Calculate Metrics
        # NOTE: pos_label=0 because 0 = Toxic. We are measuring its ability to catch the 0s!
        acc = accuracy_score(y_test, predictions)
        rec = recall_score(y_test, predictions, pos_label=0)
        prec = precision_score(y_test, predictions, pos_label=0)
        f1 = f1_score(y_test, predictions, pos_label=0)
        
        classification_results[name] = {'model': clf, 'Accuracy': acc, 'Recall': rec, 'Precision': prec, 'F1': f1}
        
        # Print the scoreboard row
        print(f"{name:<25} | {acc*100:>8.2f}% | {rec*100:>21.2f}%   | {prec*100:>24.2f}%")
        
    except Exception as e:
        print(f"{name:<25} | FAILED TO TRAIN: {str(e)[:40]}...")

🥊 THE CLASSIFICATION BAKE-OFF (CATCHING TOXIC WATER) 🥊
Algorithm Name            | Accuracy   | Recall (Caught Toxins)    | Precision (No False Alarms)
-----------------------------------------------------------------------------------------------
Linear (Logistic)         |    91.03% |                 86.67%   |                    91.23%
Distance (KNN)            |    99.31% |                 98.33%   |                   100.00%
Distance (SVM)            |    92.41% |                 83.33%   |                    98.04%
Tree (Decision Tree)      |   100.00% |                100.00%   |                   100.00%
Tree (Random Forest)      |   100.00% |                100.00%   |                   100.00%
Boosting (Gradient)       |   100.00% |                100.00%   |                   100.00%
Boosting (Hist Gradient)  |   100.00% |                100.00%   |                   100.00%
Neural Net (MLP)          |    99.31% |                 98.33%   |                   100.00%


In [3]:
import joblib
import os

# 1. Ensure the models directory exists in your project folder
os.makedirs('../models', exist_ok=True)

# 2. Extract the absolute best model from our Bake-Off dictionary
# We choose Random Forest for maximum production stability
winner_name = "Tree (Random Forest)"
best_classifier = classification_results[winner_name]['model']

# 3. Define the exact file paths for your FastAPI backend
clf_scaler_path = '../models/rw_classifier_scaler.pkl'
clf_model_path = '../models/rw_classifier.pkl'

# 4. Save the artifacts!
joblib.dump(scaler, clf_scaler_path)
joblib.dump(best_classifier, clf_model_path)

print("="*55)
print(f"🥇 Winning Algorithm: {winner_name} (100% Recall)")
print(f"📊 Classifier Saved To:   {clf_model_path}")
print(f"📏 RobustScaler Saved To: {clf_scaler_path}")



🥇 Winning Algorithm: Tree (Random Forest) (100% Recall)
📊 Classifier Saved To:   ../models/rw_classifier.pkl
📏 RobustScaler Saved To: ../models/rw_classifier_scaler.pkl
